In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. DATA LOADING AND PRE-PROCESSING
# ==========================================
print("Loading In-Sample Dataset (Training Set)...")

# Path sanitized for public repository
path_is = 'Asset_A_Asset_B_In_Sample.csv'
df_is = pd.read_csv(path_is, index_col='time', parse_dates=True)

# Column names sanitized (assuming close_A and close_B based on ingestion)
Y = df_is['close_A'].values
X = df_is['close_B'].values 

# ==========================================
# 2. KALMAN FILTER IMPLEMENTATION
# ==========================================
print("Calculating Kalman Filter...")

# Parameters sanitized for IP protection
delta = 1e-2 
wt = delta / (1 - delta) * np.eye(2)
vt = 1e-3
theta = np.zeros(2)
P = np.zeros((2, 2))
beta_array = np.zeros(len(X))

for i in range(len(X)):
    F = np.array([X[i], 1.0])
    R = P + wt
    yhat = F.dot(theta)
    et = Y[i] - yhat
    Qt = F.dot(R).dot(F) + vt
    At = R.dot(F) / Qt
    theta = theta + At * et
    P = R - np.outer(At, F).dot(R)
    beta_array[i] = theta[0]

df_is['Beta_Kalman'] = beta_array

# --- GRAPH 1: DYNAMIC HEDGE RATIO ---
plt.figure(figsize=(15, 6))
plt.plot(df_is.index, df_is['Beta_Kalman'], label='Dynamic Beta (Kalman)', color='orange')
plt.title("Dynamic Hedge Ratio (Kalman Beta)")
plt.xlabel("Date")
plt.ylabel("Beta Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# 3. STATISTICAL METRICS AND SIGNALS
# ==========================================
print("Calculating Statistical Metrics...")

df_is['Spread'] = df_is['close_A'] - (df_is['Beta_Kalman'] * df_is['close_B']) 
df_is['Spread_Mean'] = df_is['Spread'].rolling(window=500).mean()
df_is['Spread_Std'] = df_is['Spread'].rolling(window=500).std()
df_is['Z_Score'] = (df_is['Spread'] - df_is['Spread_Mean']) / df_is['Spread_Std']

# Thresholds sanitized
soglia_in = 0.0 
soglia_out = 0.0

# --- GRAPH 2: Z-SCORE AND THRESHOLDS ---
plt.figure(figsize=(15, 6))
plt.plot(df_is.index, df_is['Z_Score'], label='Spread Z-Score', color='blue', alpha=0.7)
plt.axhline(y=threshold_in, color='red', linestyle='--', label='Entry Threshold')
plt.axhline(y=-threshold_in, color='red', linestyle='--')
plt.axhline(y=threshold_out, color='green', linestyle=':', label='Exit Threshold')
plt.axhline(y=-threshold_out, color='green', linestyle=':')
plt.title("Spread Z-Score and Entry/Exit Thresholds")
plt.xlabel("Date")
plt.ylabel("Z-Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# 4. INSTITUTIONAL BACKTEST ENGINE
# ==========================================
print("Executing Institutional Backtest...")

z_scores = df_is['Z_Score'].values
positions = np.zeros(len(z_scores))
actual_position = 0

for i in range(1, len(z_scores)):
    if np.isnan(z_scores[i]): continue
    if actual_position == 0:
        if z_scores[i] < -threshold_in: actual_position = 1
        elif z_scores[i] > threshold_in: actual_position = -1
    else:
        if abs(z_scores[i]) <= soglia_out: actual_position = 0
    positions[i] = actual_position

df_is['Position'] = posizioni
df_is['Spread_Returns'] = df_is['Spread'].diff()
df_is['Strategy_Returns'] = df_is['Position'].shift(1) * df_is['Spread_Returns']

# Friction Injection: 2 pips estimated cost per round trip
COST_PER_TRADE = 0.00020
df_is['Trade_Executed'] = df_is['Position'].diff().abs() > 0
df_is['Net_Strategy_Returns'] = df_is['Strategy_Returns'].copy()
df_is.loc[df_is['Trade_Executed'], 'Net_Strategy_Returns'] -= COST_PER_TRADE

# Money Management (Sanitized lots)
INITIAL_CAPITAL = 10000.0   
LOT_SIZE = 0.0              
POINT_VALUE = 100000        

df_is['PnL_USD'] = df_is['Net_Strategy_Returns'] * POINT_VALUE * LOT_SIZE
df_is['Account_Balance'] = INITIAL_CAPITAL + df_is['PnL_USD'].cumsum()

# --- GRAPH 3: EQUITY CURVE ---
plt.figure(figsize=(15, 6))
plt.plot(df_is.index, df_is['Account_Balance'], label='Account Balance ($)', color='green')
plt.title("Strategy Equity Curve (Net of Transaction Costs)")
plt.xlabel("Date")
plt.ylabel("Balance ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# 5. FORENSIC DAILY DRAWDOWN ANALYSIS
# ==========================================
print("Executing Forensic Daily Drawdown Analysis...")

df_is['Date'] = df_is.index.date
daily_stats = df_is.groupby('Date')['Account_Balance'].agg(['first', 'min'])
daily_stats['Daily_Drawdown_Pct'] = ((daily_stats['min'] - daily_stats['first']) / daily_stats['first']) * 100

worst_dd_day = daily_stats['Daily_Drawdown_Pct'].min()

print("=" * 50)
print(f"WORST DAILY DRAWDOWN: {worst_dd_day:.2f}%")
print("=" * 50)

# --- GRAPH 4: DAILY DRAWDOWN HISTOGRAM ---
plt.figure(figsize=(15, 6))
plt.bar(daily_stats.index, daily_stats['Daily_Drawdown_Pct'], color='darkred', alpha=0.7)
plt.axhline(y=-5.0, color='black', linestyle='--', linewidth=2, label='Prop Firm Death Limit (-5%)')
plt.title("Daily Stress Test: Drawdown from Midnight")
plt.xlabel("Date")
plt.ylabel("Drawdown %")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()